# Export CheXbert prediction vectors for blinded human validation

This notebook joins the frozen blinded annotation crosswalk to the CheXbert-derived `prediction_vector` from `labeled_generation.jsonl`. It writes one CSV row per blinded report without exposing any additional generated-report content.

In [ ]:
from pathlib import Path
import json
import os
import sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR or run from the repository root")

RERUN_DIR = find_rerun_dir()
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
sys.path[:] = [entry for entry in sys.path if Path(entry or ".").resolve() != implementation_dir]
sys.path.insert(0, str(RERUN_DIR / "src"))

from rerun_code.config import load_config, output_paths
from rerun_code.common import read_jsonl

RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code root:", RERUN_DIR)
print("Results root:", PATHS["root"])


In [ ]:
import csv

LABELS_13 = [
    "atelectasis", "cardiomegaly", "consolidation", "edema",
    "enlarged cardiomediastinum", "fracture", "lung lesion", "lung opacity",
    "pleural effusion", "pleural other", "pneumonia", "pneumothorax",
    "support devices",
]
validation_dir = PATHS["labeler"] / "validation"
crosswalk_path = validation_dir / "human_annotation_crosswalk.csv"
generation_path = PATHS["labels"] / "labeled_generation.jsonl"
output_path = validation_dir / "chexbert_prediction_vectors_by_blinded_annotation_id.csv"

if not crosswalk_path.exists():
    raise FileNotFoundError(f"Frozen crosswalk is missing: {crosswalk_path}")
if not generation_path.exists():
    raise FileNotFoundError(f"Labeled generation cohort is missing: {generation_path}")

with crosswalk_path.open(encoding="utf-8-sig", newline="") as handle:
    crosswalk = list(csv.DictReader(handle))
if len(crosswalk) != 200:
    raise AssertionError(f"Expected the frozen 200-report crosswalk, found {len(crosswalk)} rows")
if len({row["blinded_annotation_id"] for row in crosswalk}) != len(crosswalk):
    raise AssertionError("Crosswalk has duplicate blinded_annotation_id values")
if len({row["generation_record_id"] for row in crosswalk}) != len(crosswalk):
    raise AssertionError("Crosswalk has duplicate generation_record_id values")

generation = read_jsonl(generation_path)
by_generation_id = {str(row["generation_record_id"]): row for row in generation}
if len(by_generation_id) != len(generation):
    raise AssertionError("Labeled generation cohort has duplicate generation_record_id values")

records = []
for row in crosswalk:
    generation_id = str(row["generation_record_id"])
    generated = by_generation_id.get(generation_id)
    if generated is None:
        raise KeyError(f"Crosswalk record is absent from the labeled cohort: {generation_id}")
    vector = generated.get("prediction_vector")
    if not isinstance(vector, list) or len(vector) != len(LABELS_13) or any(value not in (0, 1) for value in vector):
        raise ValueError(f"Invalid 13-element binary prediction_vector for {generation_id}: {vector!r}")
    output = {
        "blinded_annotation_id": row["blinded_annotation_id"],
        "generation_record_id": generation_id,
        "prediction_vector": json.dumps(vector, separators=(",", ":")),
    }
    output.update({f"chexbert_{label}": value for label, value in zip(LABELS_13, vector)})
    records.append(output)

fields = [
    "blinded_annotation_id", "generation_record_id", "prediction_vector",
    *[f"chexbert_{label}" for label in LABELS_13],
]
with output_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(records)

print(f"Saved {len(records)} vectors to: {output_path}")
print("Columns:", fields)
records[:3]
